# THYROID_2026 — Publication Figures

**Version:** v2026.03.10-publication-ready  
**Cohort:** 11,673 thyroid surgery patients | Emory University  
**Tag:** `v2026.03.10-publication-ready`

Figures produced here are intended for manuscript submission.  
Interactive HTML (Plotly) + static PNG/PDF (matplotlib) outputs are both generated.

| Figure | Title | View |
|--------|-------|------|
| 1 | AJCC 8th Stage Distribution | `survival_cohort_ready_mv` |
| 2 | Recurrence Risk by ETE Status | `risk_enriched_mv` |
| 3 | Kaplan–Meier: Recurrence-Free Survival by AJCC Stage | `survival_cohort_ready_mv` |
| 4 | Kaplan–Meier: Recurrence-Free Survival by ETE | `risk_enriched_mv` |
| 5 | Molecular Marker Co-occurrence | `risk_enriched_mv` |
| 6 | Laterality Mismatch Heatmap | `qa_laterality_mismatches` |
| 7 | Tumor Size Distribution by Histology | `advanced_features_v3` |
| 8 | Thyroglobulin Trajectory by Risk Band | `risk_enriched_mv` |

In [ ]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import duckdb
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

warnings.filterwarnings('ignore')
np.random.seed(42)

# ---------------------------------------------------------------------------
# Data loading — local DuckDB preferred; parquet bundle fallback
# ---------------------------------------------------------------------------
REPO_ROOT   = Path('..').resolve()
DB_PATH     = REPO_ROOT / 'thyroid_master_local.duckdb'
BUNDLE_DIR  = REPO_ROOT / 'exports' / 'THYROID_2026_PUBLICATION_BUNDLE_20260310_0414'
FIG_DIR     = REPO_ROOT / 'studies' / 'proposal2_ete_staging' / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

if DB_PATH.exists():
    con = duckdb.connect(str(DB_PATH), read_only=True)
    print(f'✅ Connected to local DuckDB: {DB_PATH}')

    def q(sql: str) -> pd.DataFrame:
        return con.execute(sql).fetchdf()

    risk     = q('SELECT * FROM risk_enriched_mv')
    survival = q('SELECT * FROM survival_cohort_ready_mv')
    afv3     = q('SELECT * FROM advanced_features_v3')
    lat      = q('SELECT * FROM qa_laterality_mismatches')
else:
    con = None
    print(f'⚠️  Local DuckDB not found — loading parquet bundle from {BUNDLE_DIR}')
    risk     = pd.read_parquet(BUNDLE_DIR / 'risk_enriched_mv.parquet')
    survival = pd.read_parquet(BUNDLE_DIR / 'survival_cohort_ready_mv.parquet')
    afv3     = pd.read_parquet(BUNDLE_DIR / 'advanced_features_v3.parquet')
    lat      = pd.read_parquet(REPO_ROOT / 'exports' / 'qa_laterality_mismatches.parquet')

print(f'  risk_enriched_mv        : {len(risk):,} rows')
print(f'  survival_cohort_ready_mv: {len(survival):,} rows')
print(f'  advanced_features_v3    : {len(afv3):,} rows')
print(f'  qa_laterality_mismatches: {len(lat):,} rows')

In [ ]:
# ---------------------------------------------------------------------------
# Publication theme
# ---------------------------------------------------------------------------
PALETTE = {
    'stage_I'   : '#2196F3',
    'stage_II'  : '#FF9800',
    'stage_III' : '#F44336',
    'stage_IV'  : '#9C27B0',
    'ete_pos'   : '#E53935',
    'ete_neg'   : '#1E88E5',
    'low_risk'  : '#43A047',
    'mid_risk'  : '#FB8C00',
    'high_risk' : '#E53935',
    'braf'      : '#5C6BC0',
    'ras'       : '#26A69A',
    'tert'      : '#EF5350',
}

PLOTLY_LAYOUT = dict(
    font=dict(family='Arial, sans-serif', size=13, color='#1a1a2e'),
    plot_bgcolor='white',
    paper_bgcolor='white',
    margin=dict(l=60, r=30, t=60, b=60),
    title_font_size=16,
    title_font_color='#1a1a2e',
)

STAGE_ORDER = ['I', 'II', 'III', 'IVA', 'IVB', 'IVC']
STAGE_COLORS = ['#2196F3', '#FF9800', '#F44336', '#9C27B0', '#880E4F', '#37474F']

plt.rcParams.update({
    'font.family'   : 'DejaVu Sans',
    'font.size'     : 11,
    'axes.spines.top'   : False,
    'axes.spines.right' : False,
    'axes.grid'     : True,
    'axes.grid.axis': 'y',
    'grid.alpha'    : 0.35,
    'figure.dpi'    : 150,
})

def save_fig(fig, name: str, plotly: bool = True) -> None:
    """Save Plotly figure to interactive HTML and static PNG."""
    html_path = FIG_DIR / f'{name}.html'
    png_path  = FIG_DIR / f'{name}.png'
    if plotly:
        fig.write_html(str(html_path))
        try:
            fig.write_image(str(png_path), scale=2, width=900, height=560)
            print(f'  ✅ {name}.html + .png saved')
        except Exception:
            print(f'  📄 {name}.html saved (kaleido not installed — skipping PNG)')
    else:
        plt.savefig(str(png_path), dpi=300, bbox_inches='tight')
        print(f'  ✅ {name}.png saved')
        plt.close()

print('Theme ready. Figures will be saved to:', FIG_DIR)

## Figure 1 — AJCC 8th Edition Stage Distribution

In [ ]:
df1 = (
    survival.dropna(subset=['overall_stage_ajcc8'])
    .groupby('overall_stage_ajcc8', as_index=False)
    .agg(n=('research_id', 'nunique'))
)
df1['stage_order'] = df1['overall_stage_ajcc8'].map(
    {s: i for i, s in enumerate(STAGE_ORDER)}
).fillna(99)
df1 = df1.sort_values('stage_order')
df1['pct'] = (df1['n'] / df1['n'].sum() * 100).round(1)
df1['label'] = df1['n'].map('{:,}'.format) + '<br>(' + df1['pct'].astype(str) + '%)'

fig1 = go.Figure(go.Bar(
    x=df1['overall_stage_ajcc8'],
    y=df1['n'],
    text=df1['label'],
    textposition='outside',
    marker_color=STAGE_COLORS[:len(df1)],
    marker_line_width=0,
))
fig1.update_layout(
    **PLOTLY_LAYOUT,
    title='Figure 1. AJCC 8th Edition Stage Distribution<br><sup>Thyroid Cancer Research Cohort (N={:,})</sup>'.format(df1['n'].sum()),
    xaxis_title='AJCC 8th Edition Stage',
    yaxis_title='Number of Patients',
    showlegend=False,
    yaxis_range=[0, df1['n'].max() * 1.18],
)
fig1.show()
save_fig(fig1, 'fig1_ajcc_stage_distribution')

## Figure 2 — Recurrence Risk Stratification by ETE Status

In [ ]:
df2 = (
    risk.dropna(subset=['ete', 'recurrence_risk_band'])
    .groupby(['ete', 'recurrence_risk_band'], as_index=False)
    .agg(n=('research_id', 'count'))
)
df2['ete_label'] = df2['ete'].map({True: 'ETE Present', False: 'ETE Absent'})

RISK_ORDER  = ['low', 'intermediate', 'high']
RISK_COLORS = {'low': PALETTE['low_risk'], 'intermediate': PALETTE['mid_risk'], 'high': PALETTE['high_risk']}

fig2 = px.bar(
    df2, x='ete_label', y='n', color='recurrence_risk_band',
    barmode='group',
    color_discrete_map=RISK_COLORS,
    category_orders={'recurrence_risk_band': RISK_ORDER},
    labels={'ete_label': 'ETE Status', 'n': 'Patients', 'recurrence_risk_band': 'Risk Band'},
    text_auto=True,
)
fig2.update_layout(
    **PLOTLY_LAYOUT,
    title='Figure 2. Recurrence Risk Stratification by ETE Status<br><sup>Low / Intermediate / High ATA risk bands (N={:,})</sup>'.format(df2['n'].sum()),
    legend_title='Recurrence Risk',
)
fig2.update_traces(textposition='outside')
fig2.show()
save_fig(fig2, 'fig2_ete_recurrence_risk')

## Figure 3 — Kaplan–Meier: Recurrence-Free Survival by AJCC Stage

In [ ]:
try:
    from lifelines import KaplanMeierFitter
    from lifelines.statistics import logrank_test
    HAS_LIFELINES = True
except ImportError:
    HAS_LIFELINES = False
    print('⚠️  lifelines not installed — run: pip install lifelines')

if HAS_LIFELINES:
    df3 = survival.dropna(subset=['time_to_event_days', 'event_occurred', 'overall_stage_ajcc8']).copy()
    df3['time_to_event_years'] = df3['time_to_event_days'] / 365.25

    stages_to_plot = [s for s in STAGE_ORDER if s in df3['overall_stage_ajcc8'].unique()]
    color_map = {s: c for s, c in zip(STAGE_ORDER, STAGE_COLORS)}

    fig3, ax3 = plt.subplots(figsize=(9, 6))
    kmf = KaplanMeierFitter()
    for stage in stages_to_plot:
        mask = df3['overall_stage_ajcc8'] == stage
        n = mask.sum()
        kmf.fit(
            df3.loc[mask, 'time_to_event_years'],
            event_observed=df3.loc[mask, 'event_occurred'],
            label=f'Stage {stage} (n={n:,})'
        )
        kmf.plot_survival_function(
            ax=ax3, ci_show=True, ci_alpha=0.08,
            color=color_map[stage], linewidth=2
        )

    ax3.set_xlabel('Years from Primary Surgery', fontsize=12)
    ax3.set_ylabel('Recurrence-Free Probability', fontsize=12)
    ax3.set_title(
        'Figure 3. Kaplan–Meier: Recurrence-Free Survival by AJCC 8th Stage\n'
        f'N={len(df3):,} patients with complete follow-up',
        fontsize=13
    )
    ax3.set_ylim(0, 1.05)
    ax3.legend(fontsize=9, loc='lower left', framealpha=0.8)
    plt.tight_layout()
    save_fig(None, 'fig3_km_ajcc_stage', plotly=False)
    plt.show()
else:
    print('Skipped — install lifelines')

## Figure 4 — Kaplan–Meier: Recurrence-Free Survival by ETE Status

In [ ]:
if HAS_LIFELINES:
    df4 = risk.dropna(subset=['time_to_event_days', 'event_occurred', 'ete']).copy()
    df4['time_years'] = df4['time_to_event_days'] / 365.25
    df4['ete_label']  = df4['ete'].map({True: 'ETE Present', False: 'ETE Absent'})

    # Log-rank test
    ete_pos = df4[df4['ete']]
    ete_neg = df4[~df4['ete']]
    lr = logrank_test(
        ete_pos['time_years'], ete_neg['time_years'],
        event_observed_A=ete_pos['event_occurred'],
        event_observed_B=ete_neg['event_occurred'],
    )
    p_val_str = f'p = {lr.p_value:.4f}' if lr.p_value >= 0.0001 else 'p < 0.0001'

    fig4, ax4 = plt.subplots(figsize=(8, 6))
    kmf4 = KaplanMeierFitter()
    for label, color in [('ETE Present', PALETTE['ete_pos']), ('ETE Absent', PALETTE['ete_neg'])]:
        mask = df4['ete_label'] == label
        n = mask.sum()
        kmf4.fit(
            df4.loc[mask, 'time_years'],
            event_observed=df4.loc[mask, 'event_occurred'],
            label=f'{label} (n={n:,})'
        )
        kmf4.plot_survival_function(ax=ax4, ci_show=True, ci_alpha=0.1, color=color, linewidth=2)

    ax4.text(0.05, 0.12, f'Log-rank {p_val_str}',
             transform=ax4.transAxes, fontsize=11,
             bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow', edgecolor='grey', alpha=0.8))
    ax4.set_xlabel('Years from Primary Surgery', fontsize=12)
    ax4.set_ylabel('Recurrence-Free Probability', fontsize=12)
    ax4.set_title(
        f'Figure 4. Kaplan–Meier: Recurrence-Free Survival by ETE Status\n'
        f'N={len(df4):,} patients',
        fontsize=13
    )
    ax4.set_ylim(0, 1.05)
    ax4.legend(fontsize=10, loc='lower left', framealpha=0.8)
    plt.tight_layout()
    save_fig(None, 'fig4_km_ete_status', plotly=False)
    plt.show()
    print(f'Log-rank statistic: {lr.test_statistic:.4f} | {p_val_str}')
else:
    print('Skipped — install lifelines')

## Figure 5 — Molecular Marker Co-occurrence

In [ ]:
markers = ['braf_positive', 'ras_positive', 'ret_positive', 'tert_positive']
marker_labels = {'braf_positive': 'BRAF', 'ras_positive': 'RAS',
                 'ret_positive': 'RET/PTC', 'tert_positive': 'TERT'}

df5 = risk[markers].dropna().astype(bool)
n_tested = len(df5)

# Co-occurrence matrix: fraction of marker-A positives that are also marker-B positive
comat = pd.DataFrame(index=markers, columns=markers, dtype=float)
for m1 in markers:
    for m2 in markers:
        pos_m1 = df5[m1]
        if pos_m1.sum() == 0:
            comat.loc[m1, m2] = 0.0
        else:
            comat.loc[m1, m2] = (pos_m1 & df5[m2]).sum() / pos_m1.sum()

comat.index   = [marker_labels[m] for m in markers]
comat.columns = [marker_labels[m] for m in markers]

fig5 = px.imshow(
    comat.astype(float),
    text_auto='.1%',
    color_continuous_scale='Blues',
    zmin=0, zmax=1,
    labels={'color': 'Co-occurrence Rate'},
    aspect='equal',
)
fig5.update_layout(
    **PLOTLY_LAYOUT,
    title=f'Figure 5. Molecular Marker Co-occurrence<br><sup>Fraction of row-marker positives also positive for column marker (N={n_tested:,} tested)</sup>',
    xaxis_title='',
    yaxis_title='',
    coloraxis_colorbar_title='Rate',
)
fig5.show()
save_fig(fig5, 'fig5_molecular_cooccurrence')

## Figure 6 — Laterality Mismatch Heatmap

In [ ]:
df6 = (
    lat.dropna(subset=['operative_side', 'path_side'])
    .groupby(['operative_side', 'path_side'], as_index=False)
    .size()
    .rename(columns={'size': 'count'})
)
pivot6 = df6.pivot(index='operative_side', columns='path_side', values='count').fillna(0).astype(int)

fig6 = px.imshow(
    pivot6,
    text_auto=True,
    color_continuous_scale=[
        [0.0, '#ffffff'], [0.4, '#FFF176'], [0.7, '#FF7043'], [1.0, '#B71C1C']
    ],
    labels={'color': 'Cases', 'x': 'Pathology Side', 'y': 'Operative Side'},
    aspect='equal',
)
fig6.update_layout(
    **PLOTLY_LAYOUT,
    title=(
        f'Figure 6. Laterality Mismatch: Operative vs Pathology Side<br>'
        f'<sup>N={len(lat):,} total records — diagonal = concordant, off-diagonal = mismatch</sup>'
    ),
)
fig6.show()
save_fig(fig6, 'fig6_laterality_mismatch_heatmap')

## Figure 7 — Tumor Size Distribution by Histology

In [ ]:
HISTO_KEEP = ['PTC', 'FTC', 'MTC', 'ATC', 'HCC']
df7 = afv3[
    afv3['histology_1_type'].isin(HISTO_KEEP) &
    afv3['largest_tumor_cm'].notna() &
    (afv3['largest_tumor_cm'] > 0) &
    (afv3['largest_tumor_cm'] < 15)
].copy()

HISTO_COLORS = {'PTC': '#1E88E5', 'FTC': '#43A047', 'MTC': '#FB8C00', 'ATC': '#E53935', 'HCC': '#8E24AA'}

medians = df7.groupby('histology_1_type')['largest_tumor_cm'].median().reset_index()
medians.columns = ['histology_1_type', 'median_cm']
medians['label'] = medians['median_cm'].map(lambda v: f'Median: {v:.1f} cm')

fig7 = px.violin(
    df7, x='histology_1_type', y='largest_tumor_cm',
    color='histology_1_type',
    color_discrete_map=HISTO_COLORS,
    category_orders={'histology_1_type': HISTO_KEEP},
    box=True, points=False,
    labels={'histology_1_type': 'Histology', 'largest_tumor_cm': 'Largest Tumor (cm)'},
)
# Annotate medians
for _, row in medians.iterrows():
    x_idx = HISTO_KEEP.index(row['histology_1_type']) if row['histology_1_type'] in HISTO_KEEP else None
    if x_idx is not None:
        fig7.add_annotation(
            x=row['histology_1_type'], y=row['median_cm'] + 0.5,
            text=row['label'], showarrow=False,
            font=dict(size=10, color='#222'),
        )
fig7.update_layout(
    **PLOTLY_LAYOUT,
    title=f'Figure 7. Tumor Size Distribution by Histology<br><sup>N={len(df7):,} tumors with measured size</sup>',
    showlegend=False,
    yaxis_title='Largest Tumor Diameter (cm)',
)
fig7.show()
save_fig(fig7, 'fig7_tumor_size_by_histology')

## Figure 8 — Thyroglobulin Trajectory by Recurrence Risk Band

In [ ]:
df8 = risk.dropna(subset=['recurrence_risk_band', 'tg_first', 'tg_last']).copy()
df8 = df8[(df8['tg_first'] > 0) & (df8['tg_last'] >= 0) & (df8['tg_max'] < 500)]

df8_melt = df8.melt(
    id_vars=['research_id', 'recurrence_risk_band'],
    value_vars=['tg_first', 'tg_last'],
    var_name='timepoint', value_name='tg_value'
)
df8_melt['timepoint_label'] = df8_melt['timepoint'].map({'tg_first': 'First Tg', 'tg_last': 'Last Tg'})
RISK_ORDER = ['low', 'intermediate', 'high']
df8_melt = df8_melt[df8_melt['recurrence_risk_band'].isin(RISK_ORDER)]

fig8 = px.box(
    df8_melt,
    x='recurrence_risk_band', y='tg_value',
    color='timepoint_label',
    facet_col=None,
    category_orders={'recurrence_risk_band': RISK_ORDER},
    color_discrete_map={'First Tg': '#42A5F5', 'Last Tg': '#EF5350'},
    points=False,
    labels={'recurrence_risk_band': 'Recurrence Risk Band', 'tg_value': 'Thyroglobulin (ng/mL)', 'timepoint_label': 'Timepoint'},
)
fig8.update_layout(
    **PLOTLY_LAYOUT,
    title=f'Figure 8. Thyroglobulin Trajectory by Recurrence Risk Band<br><sup>First vs Last Tg measurement (N={len(df8):,} patients with serial Tg)</sup>',
    yaxis_range=[0, df8_melt['tg_value'].quantile(0.97)],
    legend_title='Timepoint',
)
fig8.show()
save_fig(fig8, 'fig8_tg_trajectory_by_risk_band')

## Export Summary

## Timeline Explorer Example

Per-patient event timeline (surgery, labs, RAI, recurrence) from `dashboard_patient_timeline_mv` or `master_timeline`. Use this pattern to replicate the dashboard Timeline tab in a static figure or export.

In [ ]:
# Timeline Explorer example: surgery events from master_timeline (DuckDB only)
if con is not None:
    try:
        tables = [r[0] for r in con.execute("SELECT table_name FROM information_schema.tables WHERE table_schema = 'main'").fetchall()]
        if 'master_timeline' in tables:
            tl = con.execute("SELECT research_id, surgery_number, surgery_date, surgery_type FROM master_timeline ORDER BY research_id, surgery_number LIMIT 1000").fetchdf()
            tl = tl.dropna(subset=['surgery_date'])
            if not tl.empty:
                agg = tl.groupby('research_id', as_index=False).agg(n_surgeries=('surgery_number', 'max'))
                fig_tl = px.bar(agg.head(30), x='research_id', y='n_surgeries',
                    title='Timeline Explorer example — Surgery count per patient (sample)',
                    labels={'research_id': 'Patient ID', 'n_surgeries': 'Number of surgeries'})
                fig_tl.update_layout(**PLOTLY_LAYOUT)
                fig_tl.show()
                save_fig(fig_tl, 'timeline_explorer_example')
            else:
                print('No surgery dates in master_timeline.')
        else:
            print('master_timeline not found — use dashboard for full Timeline Explorer.')
    except Exception as e:
        print(f'Timeline Explorer skip: {e}')
else:
    print('Timeline Explorer requires local DuckDB (master_timeline).')

In [ ]:
import os

print('=== Publication Figures Export Summary ===')
print(f'Output directory: {FIG_DIR}\n')

figure_files = sorted(FIG_DIR.glob('fig*.{html,png}')) + sorted(FIG_DIR.glob('fig*.png')) + sorted(FIG_DIR.glob('fig*.html'))
seen = set()
for f in sorted(FIG_DIR.iterdir()):
    if f.name in seen:
        continue
    seen.add(f.name)
    size_kb = f.stat().st_size / 1024
    fmt = '🌐 HTML' if f.suffix == '.html' else '🖼️  PNG'
    print(f'  {fmt}  {f.name:<55} {size_kb:>7.1f} KB')

print(f'\n✅ All figures ready in {FIG_DIR}')
print('   Submit .png files for journal; share .html files for supplementary interactive figures.')
print(f'   Reproducibility seed: np.random.seed(42)')
print(f'   Data tag: v2026.03.10-publication-ready')

In [ ]:
# Close DB connection
if con is not None:
    con.close()
    print('DuckDB connection closed.')